# Try out the NVIDIA AI-Q Blueprint!
This notebook deploys the NVIDIA AI-Q Blueprint — a reference application for an enterprise-grade research agent built with the [NeMo Agent toolkit](https://docs.nvidia.com/nemo/agent-toolkit/latest/). It uses a two-tier research architecture that keeps simple queries fast while reserving multi-phase deep research for complex topics. The blueprint demonstrates an AI-powered research workflow that combines intent-aware routing, shallow and deep research agents, and optional human-in-the-loop clarification, plus benchmarks and evaluation harnesses for evaluation-driven development.

Some of the key capabilities of this blueprint are:

| Capability | What it Does | Value |
|------------|--------------|-------|
| Intelligence Dynamic Routing | Automatically selects the right model, workflow depth, and data sources for each request | Best performance at optimal cost |
| Custom Evaluation Metrics | Built-in evaluation harness with transparent reasoning and explainability | Increased trust, auditability, reliability |
| Open Recipe | Fully open, modular codebase with pluggable components | Easier to customize and extend |
| Persistent Context Management | Virtual file system and workspaces for storing and recalling data, code, and insights | Handles long-running workflows efficiently |

In this notebook, you will install the prerequisites, set up the workflow environment, and connect to NVIDIA-hosted NIM APIs. Once deployed, you will have a fully functional research assistant that can answer simple questions via fast tool-augmented lookup (shallow research) and produce citation-backed, publication-style reports for complex topics (deep research). You can run the blueprint with web search only, customize models and tools, and optionally add pluggable knowledge retrieval.


Note: this notebook is designed to run as a [brev.dev launchable](https://brev.nvidia.com/org/fxvddnw0d/environments/8cawjwtt0).

![Architecture diagram](../assets/AIQ-arch-light.png)

# Getting Started

> [Software Components](#software-components)  
> [Hardware Requirements](#hardware-requirements)  
> [Prerequisites](#prerequisites)  
> [Spin Up Blueprint](#spin-up-blueprint)  
> [Environment Setup](#environment-setup)  
> [Run agent with NAT](#run-this-agent-using-the-nemo-agent-toolkit-run-command)  
> [Meta / Shallow / Deep examples](#meta-responses)  
> [Deployment using Docker Compose](#deployment-using-docker-compose)  
> [Next Steps](#next-steps)

## Software Components

The following are used by this project:

- [NVIDIA NeMo Agent toolkit](https://docs.nvidia.com/nemo/agent-toolkit/latest/)
- [NIM of nvidia/nemotron-3-nano-30b-a3b](https://build.nvidia.com/nvidia/nemotron-3-nano-30b-a3b)
- [NIM of nvidia/llama-3_2-nv-embedqa-1b-v2](https://build.nvidia.com/nvidia/llama-3_2-nv-embedqa-1b-v2) (Optional)
- [NIM of nvidia/nemotron-nano-12b-v2-vl](https://build.nvidia.com/nvidia/nemotron-nano-12b-v2-vl) (Optional)
- [NVIDIA nvidia/nemotron-mini-4b-instruct](https://build.nvidia.com/nvidia/nemotron-mini-4b-instruct) (Optional)
- [Tavily Search API](https://tavily.com/) for web search

## Hardware Requirements

We have two deployment options: 

- **Using Hosted NIMs**:
If you are deploying this blueprint without the NVIDIA RAG knowledge layer backend, there are **no GPU requirements** for this blueprint on its own.

- **Self-hosting NIMs:**
To deploy the NIM for `nvidia/nemotron-3-nano-30b-a3b`, see [model card](https://build.nvidia.com/nvidia/nemotron-3-nano-30b-a3b/modelcard).  
If using NVIDIA RAG knowledge layer backend, please see [NVIDIA RAG Documentation Pages](https://docs.nvidia.com/rag/latest/support-matrix.html)



## Prerequisites

- **Docker Compose**
- **LLM API key** – For your chosen provider (required for LLM inference):
    - **NVIDIA API Key** from [NVIDIA AI](https://build.nvidia.com/) (for NIM models)
Optional requirements:
- **Web search API key**:
  - **Tavily** – [tavily.com](https://tavily.com) (for general web search)
- **Node.js 18+ and npm** (optional, for web UI mode)

## Spin Up Blueprint

We will be using docker compose to deploy this blueprint and interact with the NAT `run` command as well as through the custom chat-based frontend UI.

### Environment Setup

1. **Clone** the NVIDIA AI-Q Blueprint workflow repository
>**Note**: Skip this step if you are not on brev and instead running from within the repository.

In [ ]:
%%bash
git clone https://github.com/NVIDIA-AI-Blueprints/aiq.git
cd aiq

2. **Setup** Environment

In [ ]:
# Run setup from the project root (where pyproject.toml and scripts/setup.sh live).
import os
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "scripts" / "setup.sh").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise FileNotFoundError("Could not locate project root containing pyproject.toml and scripts/setup.sh")

os.environ["REPO_ROOT"] = str(REPO_ROOT)
os.chdir(REPO_ROOT)
os.environ["UV_VENV_CLEAR"] = "1"
get_ipython().system("./scripts/setup.sh")

In [ ]:
%%bash
source .venv/bin/activate

3. **Obtain and set** API keys and environment variables

| API | Environment variable | Purpose |
|-----|----------------------|--------|
| **NVIDIA** | `NVIDIA_API_KEY` | NIM / NGC model inference (required) |
| **Tavily** | `TAVILY_API_KEY` | Web search (recommended) |

**NVIDIA API key:**  
1. Go to [NVIDIA Build](https://build.nvidia.com/explore/discover) or [NGC](https://ngc.nvidia.com/).  
2. Sign in and open a model card (e.g. Nemotron).  
3. Use **Get API Key** → **Generate Key**, then copy the key (starts with `nvapi-`).  

**Tavily API key:**  
1. Go to [tavily.com](https://tavily.com) and create an account.  
2. Create an API key in the dashboard and add it to your environment.  

In [ ]:
import getpass
import os

if "NVIDIA_API_KEY" not in os.environ:
    nvidia_api_key = getpass.getpass("Enter your NVIDIA API key: ")
    os.environ["NVIDIA_API_KEY"] = nvidia_api_key

if "TAVILY_API_KEY" not in os.environ:
    tavily_api_key = getpass.getpass("Enter your Tavily API key: ")
    os.environ["TAVILY_API_KEY"] = tavily_api_key

### Run this agent using the NeMo Agent Toolkit [`run` command](https://docs.nvidia.com/nemo/agent-toolkit/latest/run-workflows/about-running-workflows.html#using-the-nat-run-command)

In [ ]:
%%writefile config_simple_researcher.yml

general:
  telemetry:
    logging:
      console:
        _type: console
        level: INFO

llms:
  nemotron_llm_intent:
    _type: nim
    model_name: nvidia/nemotron-3-nano-30b-a3b
    base_url: "https://integrate.api.nvidia.com/v1"
    temperature: 0.5
    top_p: 0.9
    max_tokens: 4096
    num_retries: 5
    chat_template_kwargs:
      enable_thinking: true

  nemotron_llm:
    _type: nim
    model_name: nvidia/nemotron-3-nano-30b-a3b
    base_url: "https://integrate.api.nvidia.com/v1"
    temperature: 0.1
    top_p: 0.3
    max_tokens: 16384
    num_retries: 5
    chat_template_kwargs:
      enable_thinking: true

  nemotron_llm_deep:
    _type: nim
    model_name: nvidia/nemotron-3-nano-30b-a3b
    base_url: "https://integrate.api.nvidia.com/v1"
    temperature: 1.0
    top_p: 1.0
    max_tokens: 128000
    num_retries: 5
    chat_template_kwargs:
      enable_thinking: true

functions:
  web_search_tool:
    _type: tavily_web_search
    max_results: 5
    max_content_length: 1000

  advanced_web_search_tool:
    _type: tavily_web_search
    max_results: 2
    advanced_search: true

  intent_classifier:
    _type: intent_classifier
    llm: nemotron_llm_intent
    tools:
      - web_search_tool

  shallow_research_agent:
    _type: shallow_research_agent
    llm: nemotron_llm
    tools:
      - web_search_tool

  deep_research_agent:
    _type: deep_research_agent
    orchestrator_llm: nemotron_llm_deep
    max_loops: 2
    tools:
      - advanced_web_search_tool

workflow:
  _type: chat_deepresearcher_agent
  interactive_auth: false
  checkpoint_db: ${AIQ_CHECKPOINT_DB:-./checkpoints.db}

**What this config does:** 

This config defines a NAT workflow for the AI-Q chat researcher. 

**`general`** turns on console logging. 

**`general.front_end`** wires the Web API (AI-Q API Worker), async job store, and CORS for the chat UI.

**`llms`** wires the models for the agents. All use Nemotron Nano V3 NIM, but with different settings for extracting the best performance.

**`functions`** register tools (Tavily web search, plus advanced search), the intent classifier, the clarifier agent, and the shallow and deep research agents with their assigned tools and LLMs.
 
**`workflow`** is set to `chat_deepresearcher_agent`. See source code for this full agent [here](../../src/aiq_agent/agents/chat_researcher/agent.py).

**Optional settings:** 

`enable_clarifier: true` lets the agent ask clarifying questions before deep research; 
`enable_plan_approval: true` (on the clarifier) lets users approve or adjust the research plan. 

>**Note:** the above human-in-the-loop interactions may not work in a notebook environment; you can experience these using the AI-Q CLI or the web UI deployed later in this notebook.

**Main agents:** intent classifier, clarifier, shallow research, deep research. 
**Tools:** web search tool for shallow researcher, advanced web search for deep research, and knowledge retrieval (using the LlamaIndex backend for user-uploaded documents).


#### Meta Responses 

>[Documentation](../../docs/source/architecture/agents/intent-classifier.md)

In [ ]:
!.venv/bin/nat run --config_file config_simple_researcher.yml --input "Hello, what can you do?"

#### Shallow Research
>[Documentation](../../docs/source/architecture/agents/shallow-researcher.md)

In [ ]:
!.venv/bin/nat run --config_file config_simple_researcher.yml --input "What is NVIDIA Spectrum-X primarily designed for?"

#### Deep Research 

>[Documentation](../../docs/source/architecture/agents/deep-researcher.md)  
<div class="alert alert-block alert-info">
<b>Note:</b> Deep research can take several minutes: multiple LLM calls and web searches run in sequence. Watch the logs to see planning and research steps.
</div>

In [ ]:
!.venv/bin/nat run --config_file config_simple_researcher.yml --input "Conduct a comprehensive technical comparison between NVIDIA GB300 and Google Ironwood AI accelerator platforms. Generate a structured report with the following sections: Compute Architecture, Memory Subsystem, Interconnect & Scalability, System & Power, Software Ecosystem and Benchmarks"

### Deployment using Docker Compose

1. Set environment variables for docker compose deployment

In [ ]:
os.environ["BACKEND_URL"] = "http://aiq-agent:8000"
os.environ["REQUIRE_AUTH"] = "false"
os.environ["BACKEND_CONFIG"] = (
    "/app/configs/config_web_default_llamaindex.yml"  # path to the NAT config file from inside the docker container
)

2. Create an env file with secrets

In [ ]:
%%bash
cp deploy/.env.example deploy/.env

Replace the placeholders in the created `.env` file

3. Deploy the containers

In [ ]:
!docker compose -f deploy/compose/docker-compose.yaml up -d --build

4. Verify the containers are running

In [ ]:
# Confirm all the below mentioned containers are running.
import subprocess

result = subprocess.run(
    ["docker", "ps", "--format", "table {{.ID}}\t{{.Names}}\t{{.Status}}"],
    check=False,
    capture_output=True,
    text=True,
)

print(result.stdout)

The output should look something like this:

```
CONTAINER ID   NAMES                    STATUS
5f5bf0e2438f   aiq-blueprint-ui         Up 2 minutes (healthy)
e11fad0dba6e   aiq-agent                Up 2 minutes
66d55656299b   aiq-postgres             Up 3 minutes (healthy)
```

At this point, you should be able to access the NVIDIA AI-Q frontend web application by visiting http://localhost:3000.

>**Tip:** If you are running this notebook on brev, you will need to make the port for the AI-Q frontend accessible. On the settings page for your machine, navigate to "Using Ports", enter "3000", click "Expose Port", and then click "I accept".

**Select data sources:** In the UI, open the **data sources** panel (e.g. from the right side or connections icon). You will see available sources (e.g. Web Search, and optionally Paper Search, Knowledge Layer, or enterprise sources if configured). Toggle **on** the sources you want the agent to use for the next query. Web Search is usually enabled by default.

**Send a query:** Type your question or research request in the chat input and send it. The agent will use only the **enabled** data sources for that query. For deep research, the UI may show clarification or plan approval steps; respond as prompted. Results and the final report will appear in the chat.

In [ ]:
# Bring docker containers down
!docker compose -f deploy/compose/docker-compose.yaml down -v

## Next Steps

You have now deployed the NVIDIA AI-Q Blueprint and interacted with the chat-based agent. To continue this journey, checkout the [Github repo](https://github.com/NVIDIA-AI-Blueprints/aiq).

**More notebooks**: We have walkthroughs of the deep researcher agent and its customization (see [notebooks](../notebooks/)).

**Customization Guide**: Checkout the [customization guide](../source/customization/)